<a target="_blank" href="https://colab.research.google.com/github/Tensor-Reloaded/Neural-Networks-Template-2025/blob/main/Lab02/Assignment1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Assignment 1 (10 points)**

## **Solving a linear system in python**

In this homework, you will familiarize yourself with key linear algebra con-
cepts and Python programming by solving a system of linear equations. You
will explore multiple methods for solving such systems, including Cramer’s rule
and matrix inversion. By the end of this assignment, you will have a good un-
derstanding of how to represent and manipulate matrices and vectors in Python.

We begin with the following system of 3 linear equations with 3 unknowns:
$$ 2x + 3y - z = 5 $$
$$ x - y + 4z = 6 $$
$$ 3x + y + 2z = 7 $$

This system can be vectorized in the following form:
$$ A \cdot X = B $$
where:
$$
A = \begin{bmatrix}
2 & 3 & -1 \\
1 & -1 & 4 \\
3 & 1 & 2
\end{bmatrix}, \quad
X = \begin{bmatrix}
x \\
y \\
z
\end{bmatrix}, \quad
B = \begin{bmatrix}
5 \\
6 \\
7
\end{bmatrix}
$$

**Considerations**
- do not use any linear algebra framework such as $numpy$
- use python lists as data structures for matrices and vectors
- experiment with other values for the coefficients and free terms

### **1. Parsing the System of Equations (1 point)**

The first task is to implement a Python script that reads a system of linear equations from a text file and parses it into a matrix $A$ and a vector $B$. You will use the input format described below to extract the coefficients for $A$ and $B$.

**Input File Format**
```text
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7
```

Note that the coefficients are always in the order x, y and z and the terms are always space separated

In [4]:
import pathlib
import re

def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    """
    Reads a system of 3 linear equations from a file and parses it into
    matrix A (coefficients) and vector B (constants).
    """
    coefficient_matrix = []
    constant_vector = []

    with open(path, 'r') as file:
        lines = file.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Split equation into left hand side (LHS) and right hand side (RHS)
        left_hand_side, right_hand_side = line.split('=')

        # Extract the constant term (B)
        constant_value = float(right_hand_side.strip())
        constant_vector.append(constant_value)

        # Prepare LHS to parse terms easily by ensuring signed terms are space-separated
        # e.g., '2x + 3y - z' -> ['2x', '+', '3y', '-', 'z']
        lhs_cleaned = left_hand_side.replace('+', ' + ').replace('-', ' - ')
        tokens = lhs_cleaned.split()

        coef_x, coef_y, coef_z = 0.0, 0.0, 0.0
        current_sign = 1.0

        for token in tokens:
            if token == '+':
                current_sign = 1.0
            elif token == '-':
                current_sign = -1.0
            else:
                # Parse the variable and its coefficient
                match = re.match(r'([0-9.]*)([xyz])', token)
                if match:
                    coeff_str, var_name = match.groups()
                    # If coefficient string is empty, the coefficient is 1 (e.g., 'x' or '-z')
                    val = float(coeff_str) if coeff_str else 1.0
                    signed_val = current_sign * val

                    if var_name == 'x':
                        coef_x = signed_val
                    elif var_name == 'y':
                        coef_y = signed_val
                    elif var_name == 'z':
                        coef_z = signed_val

        coefficient_matrix.append([coef_x, coef_y, coef_z])

    return coefficient_matrix, constant_vector

# Prepare dummy 'system.txt' if not exists for demo purposes
system_file_path = pathlib.Path("system.txt")
if not system_file_path.exists():
    with open(system_file_path, "w") as f:
        f.write("2x + 3y - z = 5\n")
        f.write("x - y + 4z = 6\n")
        f.write("3x + y + 2z = 7\n")

A, B = load_system(system_file_path)
print(f"{A=} {B=}")

A=[[2.0, 3.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.0, 2.0]] B=[5.0, 6.0, 7.0]


### **2. Matrix and Vector Operations (5 points)**

Once you have successfully parsed the matrix and vector, complete the following exercises to manipulate and understand basic matrix and vector operations. Write Python functions for each of these tasks:

#### 2.1. Determinant

Write a function to compute the determinant of matrix $A$. Recall one of the formulae for the determinant of a $3x3$ matrix:
$$ \text{det}(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31}) $$

In [5]:
def determinant(matrix: list[list[float]]) -> float:
    """
    Computes the determinant of a 3x3 matrix.
    """
    a11, a12, a13 = matrix[0]
    a21, a22, a23 = matrix[1]
    a31, a32, a33 = matrix[2]

    det_val = a11 * (a22 * a33 - a23 * a32) - a12 * (a21 * a33 - a23 * a31) + a13 * (a21 * a32 - a22 * a31)
    return det_val

print(f"{determinant(A)=}")

determinant(A)=14.0


#### 2.2. Trace

Compute the sum of the elements along the main diagonal of matrix $A$. For a matrix $A$, this is:
$$ \text{Trace}(A) = a_{11} + a_{22} + a_{33} $$

In [6]:
def trace(matrix: list[list[float]]) -> float:
    """
    Computes the sum of elements along the main diagonal of a square matrix.
    """
    diagonal_sum = 0.0
    for i in range(len(matrix)):
        diagonal_sum += matrix[i][i]
    return diagonal_sum

print(f"{trace(A)=}")

trace(A)=3.0


#### 2.3. Vector norm

Compute the Euclidean norm of vector $B$, which is:
$$ ||B|| = \sqrt{b_1^2 + b_2^2 + b_3^2} $$

In [7]:
import math

def norm(vector: list[float]) -> float:
    """
    Computes the Euclidean norm (L2 norm) of a vector.
    """
    squared_sum = sum(element ** 2 for element in vector)
    return math.sqrt(squared_sum)

print(f"{norm(B)=}")

norm(B)=10.488088481701515


#### 2.4. Transpose of matrix

Write a function to compute the transpose of matrix $A$. The transpose of a matrix $A$ is obtained by swapping its rows and columns.
    

In [8]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    """
    Computes the transpose of a 3x3 matrix.
    """
    num_rows = len(matrix)
    num_cols = len(matrix[0])

    transposed_matrix = [[0.0] * num_rows for _ in range(num_cols)]
    for row_idx in range(num_rows):
        for col_idx in range(num_cols):
            transposed_matrix[col_idx][row_idx] = matrix[row_idx][col_idx]

    return transposed_matrix

print(f"{transpose(A)=}")

transpose(A)=[[2.0, 1.0, 3.0], [3.0, -1.0, 1.0], [-1.0, 4.0, 2.0]]


#### 2.5. Matrix-vector multiplication

Write a function that multiplies matrix $A$ with vector $B$.

In [9]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    """
    Multiplies a 3x3 matrix with a 3x1 vector.
    """
    result_vector = []
    for row in matrix:
        row_dot_product = sum(row[col_idx] * vector[col_idx] for col_idx in range(len(vector)))
        result_vector.append(row_dot_product)
    return result_vector

print(f"{multiply(A, B)=}")

multiply(A, B)=[21.0, 27.0, 35.0]


### **3. Solving using Cramer's Rule (1 point)**

Now that you have explored basic matrix operations, solve the system of linear equations using Cramer's rule.

**Cramer's Rule:**

Cramer's rule allows you to solve for each unknown $x$, $y$, and $z$ using determinants. For example:
$$ x = \frac{\text{det}(A_x)}{\text{det}(A)}, \quad y = \frac{\text{det}(A_y)}{\text{det}(A)}, \quad z = \frac{\text{det}(A_z)}{\text{det}(A)} $$
where $A_x$, $A_y$, and $A_z$ are matrices formed by replacing the respective column of matrix $A$ with vector $B$.

In [10]:
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    """
    Solves a 3x3 system of linear equations using Cramer's Rule.
    """
    main_det = determinant(matrix)
    if main_det == 0:
        raise ValueError("The determinant of the coefficient matrix is zero; the system cannot be solved via Cramer's Rule.")

    solutions = []
    for col_idx in range(3):
        # Create a copy of the matrix with the col_idx column replaced by the B vector
        modified_matrix = []
        for row_idx in range(3):
            new_row = list(matrix[row_idx])
            new_row[col_idx] = vector[row_idx]
            modified_matrix.append(new_row)

        solutions.append(determinant(modified_matrix) / main_det)

    return solutions

print(f"{solve_cramer(A, B)=}")

solve_cramer(A, B)=[0.35714285714285715, 2.0714285714285716, 1.9285714285714286]


### **4. Solving using Inversion (3 points)**

Finally, solve the system by computing the inverse of matrix $A$ and multiplying it by vector $B$.
$$ A \cdot X = B \rightarrow X = A^{-1} \cdot B $$
**Adjugate Method for Matrix Inversion:**

To find the inverse of matrix $ A $, you can use the adjugate method:
$$ A^{-1} = \frac{1}{\text{det}(A)} \times \text{adj}(A) $$
where $\text{adj}(A)$ is the adjugate (or adjoint) matrix, which is the transpose of the cofactor matrix of $ A $.

**Cofactor Matrix:**

The cofactor matrix is a matrix where each element is replaced by its cofactor. The cofactor of an element $a_{ij}$ is given by:
$$ (-1)^{i+j} \times \text{det}(M_{ij}) $$
where $M_{ij}$ is the minor of element $a_{ij}$, which is the matrix obtained by removing the $i$-th row and $j$-th column from matrix $A$.

In [11]:
def minor(matrix: list[list[float]], target_row: int, target_col: int) -> list[list[float]]:
    """
    Returns the submatrix obtained by removing specified row and column.
    """
    sub_matrix = []
    for row_idx in range(len(matrix)):
        if row_idx == target_row:
            continue
        new_row = []
        for col_idx in range(len(matrix[row_idx])):
            if col_idx == target_col:
                continue
            new_row.append(matrix[row_idx][col_idx])
        sub_matrix.append(new_row)
    return sub_matrix

def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    """
    Computes the cofactor matrix of a 3x3 matrix.
    """
    cofactor_matrix = [[0.0] * 3 for _ in range(3)]
    for row_idx in range(3):
        for col_idx in range(3):
            sub = minor(matrix, row_idx, col_idx)
            # Determinant of 2x2 submatrix
            sub_det = sub[0][0] * sub[1][1] - sub[0][1] * sub[1][0]
            sign = (-1) ** (row_idx + col_idx)
            cofactor_matrix[row_idx][col_idx] = sign * sub_det
    return cofactor_matrix

def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    """
    The adjoint matrix is the transpose of the cofactor matrix.
    """
    cof_mat = cofactor(matrix)
    return transpose(cof_mat)

def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    """
    Solves the system AX = B using the matrix inversion method: X = (A^-1) * B.
    """
    main_det = determinant(matrix)
    if main_det == 0:
        raise ValueError("Matrix is singular and cannot be inverted.")

    adj_matrix = adjoint(matrix)

    # Invert the matrix by multiplying elements of adjoint by 1/determinant
    inverse_matrix = [[adj_matrix[r][c] / main_det for c in range(3)] for r in range(3)]

    # Solve by multiplying inverse_matrix with vector B
    return multiply(inverse_matrix, vector)

print(f"{solve(A, B)=}")

solve(A, B)=[0.3571428571428572, 2.0714285714285716, 1.9285714285714284]
